In [1]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name} Results:")
    print(f"MAE: {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²: {r2:.3f}\n")
    return mae, rmse, r2

# Safely evaluate only if the required variables exist to avoid NameError in the notebook
def _safe_eval(name, y_true_var, y_pred_var):
    g = globals()
    if y_true_var in g and y_pred_var in g:
        return evaluate_model(name, g[y_true_var], g[y_pred_var])
    else:
        missing = [v for v in (y_true_var, y_pred_var) if v not in g]
        print(f"Skipping {name}: missing variables {missing}")
        return None

_safe_eval("Linear Regression", "y_test", "y_pred_lr")
_safe_eval("Random Forest", "y_test", "y_pred_rf")


Skipping Linear Regression: missing variables ['y_test', 'y_pred_lr']
Skipping Random Forest: missing variables ['y_test', 'y_pred_rf']


In [2]:
# Complete pipeline: create X,y, split, train LR & RF, predict, evaluate
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the dataset.
# TODO: Make sure to update the path below to point to your actual processed dataset.
data = pd.read_csv("../data/processed/processed_co2_data.csv")

# --- sanity checks
if 'data' not in globals():
    raise RuntimeError("Variable `data` not found. Load your dataset into `data` first (e.g., pd.read_csv).")

print("data.shape:", data.shape)
if data.shape[0] == 0:
    raise RuntimeError("Dataset empty. Fix loading/filters before training.")

# --- prepare X and y
target = "annual co₂ emissions (tonnes )"
if target not in data.columns:
    raise KeyError(f"Target '{target}' missing. Found columns: {data.columns.tolist()}")

X = data.drop(columns=[target]).copy()
y = data[target].copy()

# Drop rows with missing target (safe)
mask = y.notna()
X = X[mask]
y = y[mask]
print("After dropping missing target ->", X.shape, y.shape)

# Handle missing values in features (X) by filling with 0.
X.fillna(0, inplace=True)

# Encode categorical columns (object/category)
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
if cat_cols:
    for col in cat_cols:
        X[col] = X[col].astype(str)  # handle NaNs/mixed types safely
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])

# Ensure numeric types for X (coerce if necessary)
# (optional) X = X.apply(pd.to_numeric, errors='coerce')

n_samples = X.shape[0]
if n_samples < 2:
    raise RuntimeError("Need at least 2 samples to perform a train/test split and evaluation.")

# Safe test_size calculation to ensure at least 1 train and 1 test sample
test_count = max(1, int(np.ceil(0.2 * n_samples)))
train_count = n_samples - test_count
if train_count < 1:
    test_count = 1
    train_count = n_samples - 1
test_size = test_count / n_samples

print(f"n_samples={n_samples}, train={train_count}, test={test_count}, test_size={test_size:.3f}")

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
print("X_train.shape:", X_train.shape, "X_test.shape:", X_test.shape)

# --- Train Linear Regression
if X_train.shape[0] > 0 and y_train.shape[0] > 0:
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    if X_test.shape[0] > 0:
        y_pred_lr = lr.predict(X_test)
    else:
        y_pred_lr = None
        print("LR trained but X_test empty.")
else:
    y_pred_lr = None
    print("Skipping LR training: empty train set.")

# --- Train Random Forest
if X_train.shape[0] > 0 and y_train.shape[0] > 0:
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    if X_test.shape[0] > 0:
        y_pred_rf = rf.predict(X_test)
    else:
        y_pred_rf = None
        print("RF trained but X_test empty.")
else:
    y_pred_rf = None
    print("Skipping RF training: empty train set.")

# --- Evaluate using your helper functions (they rely on globals)
_safe_eval("Linear Regression", "y_test", "y_pred_lr")
_safe_eval("Random Forest", "y_test", "y_pred_rf")


data.shape: (18646, 8)
After dropping missing target -> (18646, 7) (18646,)
n_samples=18646, train=14916, test=3730, test_size=0.200
X_train.shape: (14916, 7) X_test.shape: (3730, 7)
Linear Regression Results:
MAE: 306152187.923
RMSE: 1249639634.596
R²: 0.024

Random Forest Results:
MAE: 10571441.628
RMSE: 69767781.074
R²: 0.997



(10571441.627556782, np.float64(69767781.07448864), 0.9969574696764579)